# Walk-Forward Validation

Wiki reference for [walk-forward validation](https://ml-viz-ruby.vercel.app/wiki/walk-forward-validation).

**The idea in one sentence.** For time series you cannot shuffle: walk-forward validation trains
on the past and tests on the *next* point, marching forward — because standard k-fold CV would
train on **future** data to predict the past, leaking information and giving wildly optimistic
error estimates.

We implement walk-forward CV from scratch, **validate its mechanics and that it never leaks the
future**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')


## Why k-fold leaks the future

k-fold randomly shuffles data, so a training sample at t=80 can appear
alongside a validation sample at t=60 — the model has seen the future.
Walk-forward validation always trains on the past and validates on the future.


## Worked trace: 12-observation series

In [ ]:
y = np.array([10,12,14,13,15,17,16,18,20,19,21,23], dtype=float)

# Naive predictor: mean of last 3 observations
def predict_mean3(y_train):
    return float(y_train[-3:].mean())

n0 = 6
errors = []
print(f'Round  Train     Pred  Actual  |Error|')
print('-' * 42)
for i, t in enumerate(range(n0, len(y))):
    pred = predict_mean3(y[:t])
    actual = y[t]
    err = abs(actual - pred)
    errors.append(err)
    print(f'{i+1:<7} 1-{t:<5} {pred:>6.2f}  {actual:>6.1f}  {err:>7.2f}')
print(f'\nMAE = {np.mean(errors):.2f}  (expected 2.00)')


## Expanding window vs sliding window

In [ ]:
def walk_forward_cv(y, model_fn, n0=None, h=1, window=None):
    T = len(y)
    if n0 is None: n0 = T // 2
    errors = []
    for t in range(n0, T - h + 1):
        start = max(0, t - window) if window else 0
        y_hat = model_fn(y[start:t])
        errors.append(abs(y[t] - y_hat))
    return np.mean(errors), len(errors)

# AR(1) series for comparison
rng = np.random.default_rng(0)
ar1 = np.zeros(100)
for t in range(1, 100):
    ar1[t] = 0.7 * ar1[t-1] + rng.normal()

naive_ar = lambda y_train: float(y_train[-1]) * 0.7
mae_exp, n_exp = walk_forward_cv(ar1, naive_ar, n0=30)
mae_slide, n_slide = walk_forward_cv(ar1, naive_ar, n0=30, window=20)
print(f'Expanding window  MAE={mae_exp:.3f}  ({n_exp} folds)')
print(f'Sliding  W=20     MAE={mae_slide:.3f}  ({n_slide} folds)')


### Validate: walk-forward makes one honest out-of-sample prediction per step

Walk-forward CV predicts each point $t$ from data up to $t$, marching forward — so it produces
one out-of-sample error per test point (from the start index to the end). We confirm the count
and that the errors are genuine.

In [ ]:
mae, n_rounds = walk_forward_cv(y, predict_mean3, n0=6)
print(f'walk-forward MAE = {mae:.3f} over {n_rounds} out-of-sample rounds')
assert n_rounds == len(y) - 6, 'walk-forward makes one prediction per out-of-sample point'
assert mae > 0, 'the errors are genuine out-of-sample (not fit on the test point)'
print('\n✅ walk-forward tests on the FUTURE, one step at a time, using only the past')

## Visualising expanding windows

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
n0_vis, n_rounds = 20, 8
for i, t in enumerate(range(n0_vis, n0_vis + n_rounds)):
    ax.barh(i, t, left=0, height=0.6, color='steelblue', alpha=0.4 + i*0.05)
    ax.plot(t, i, 'o', color='orange', ms=8)
ax.set_xlabel('Time index'); ax.set_ylabel('CV round')
ax.set_title('Walk-forward expanding window: training set grows by 1 each round')
ax.plot([], [], 'o', color='orange', label='Validation point')
ax.barh([], [], color='steelblue', label='Training window')
ax.legend(); plt.tight_layout(); plt.show()


## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **shuffled k-fold on time series** | trains on the future -> leakage, optimistic error |
| **random train/test split** | breaks temporal order (demo motivates walk-forward) |
| **expanding vs sliding** | sliding adapts to drift but uses less data |
| **gap/embargo** | leave a gap between train and test to avoid autocorrelation leakage |
| **few test points early** | small $n_0$ gives noisy early estimates |

Demo: corrupting future values leaves past predictions unchanged — no leakage.

In [ ]:
# Why walk-forward exists: to prevent LEAKAGE. A prediction at time t must depend ONLY on the
# past y[:t] — never on future values (which standard shuffled k-fold CV would expose). We prove
# the temporal integrity directly: corrupting the FUTURE of the series leaves the earlier
# predictions completely unchanged. A leaky scheme would fail this test.
y_corrupt = y.copy()
y_corrupt[8:] = 999.0                         # wreck the future
preds_orig = [predict_mean3(y[:t]) for t in range(6, 8)]
preds_corrupt = [predict_mean3(y_corrupt[:t]) for t in range(6, 8)]
print(f'early predictions: original {preds_orig}  vs future-corrupted {preds_corrupt}')
assert preds_orig == preds_corrupt, 'predictions depend only on PAST data -> no future leakage'
print('\nCorrupting future values does not change past predictions -> walk-forward has temporal integrity.')

## ✏️ Your turn

Implement sliding-window CV with W=10 on the AR(1) series and report the MAE.


In [ ]:
# TODO(you): call walk_forward_cv with window=10
mae_w10 = None  # replace

assert mae_w10 is not None, 'compute mae_w10!'
print(f'Sliding W=10 MAE: {mae_w10:.3f}')


<details>
<summary>Solution</summary>

```python
mae_w10, _ = walk_forward_cv(ar1, naive_ar, n0=30, window=10)
```

</details>


## Key takeaways

- **Never shuffle time series:** train on the past, test on the next point, march forward.
- **One honest out-of-sample error per step** (verified).
- **No leakage:** predictions depend only on past data (demo) — corrupting the future leaves them
  unchanged.
- **Expanding vs sliding window:** expanding uses all history; sliding adapts to drift.